In [ ]:
# 라이브러리 설치

!pip install trl transformers peft bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.8 MB/s eta 0:00:00


In [ ]:
# trl 버전 확인

import trl
print(trl.__version__)

1.6.0


In [ ]:
# 파인튜닝 관련 라이브러리 임포트

import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel

from trl import SFTTrainer, SFTConfig
from sklearn.model_selection import train_test_split

In [ ]:
# 구글 드라이브 마운트 후 CSV 데이터 로드 및 확인

from google.colab import drive
drive.mount('/content/drive')

train = pd.read_csv("/content/drive/MyDrive/dataset/train.csv", encoding='utf-8-sig')
test =  pd.read_csv("/content/drive/MyDrive/dataset/test.csv", encoding='utf-8-sig')

print('Train shape:', train.shape )
print('Test shape:', test.shape )
print(train.head())

Mounted at /content/drive
Train shape: (11263, 3)
Test shape: (1689, 2)
            ID                                              input  \
0  TRAIN_00000  별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈...   
1  TRAIN_00001                             잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ   
2  TRAIN_00002                                    절테 간면 않 된는 굣 멥몫   
3  TRAIN_00003  야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵...   
4  TRAIN_00004  집윈 축쳐눌료 딴너왓눈뎁 카셩뷔 좋곱 칼쿰한네올. 쩌럼한뒈 뮬콰 욺료토 잊쿄 빻토 ...   

                                              output  
0  별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자...  
1                             잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ  
2                                    절대 가면 안 되는 곳 메모  
3  아... 가격 좋고 뷰도 뻥 뚫려서 시원하지만 담배 냄새 미쳐버림. 싸게 하루만 묵...  
4  지인 추천으로 다녀왔는데 가성비 좋고 깔끔하네요. 저렴한데 물과 음료도 있고 방도 ...  


In [ ]:
# 학습/검증 데이터 9:1 비율로 분리

from sklearn.model_selection import train_test_split

# TODO: 아래 코드를 완성하세요. (힌트: test_size는 0.1~0.2, random_state=42 고정)
train_data, val_data =  train_test_split(train, test_size=0.1, random_state=42)
print('train_data:', len(train_data))
print('val_data:', len(val_data))

train_data: 10136
val_data: 1127


In [ ]:
# Gemma-Ko-2B 모델을 4bit 양자화로 로드

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = 'beomi/gemma-ko-2b'

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
def create_prompt(input_text, output_text='', few_shot=False):
    system_prompt = (
        "당신은 난독화된 한국어 리뷰를 자연스럽고 정확한 문장으로 복원하는 전문가이다.\n"
        "반드시 복원된 문장만 출력하라.\n\n"
    )

    if few_shot:
        system_prompt += (
            "Example 1:\nInput: 배쏭 개빠름 진짜 죠아용\nOutput: 배송 개빠름 진짜 좋아용\n\n"
            "Example 2:\nInput: 마싯구 양 많아여!!\nOutput: 맛있고 양 많아요!!\n\n"
        )

    prompt = f"{system_prompt}Input: {input_text}\nOutput: "

    if output_text:
        prompt += output_text + tokenizer.eos_token

    return prompt

In [ ]:
# Gemma-Ko-2B 토크나이저 로드 및 패딩 설정

from transformers import AutoTokenizer

model_id = 'beomi/gemma-ko-2b'

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

In [ ]:
# 파인튜닝 전 모델 추론 결과 확인

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device_map='auto'
)

print('=== 파인튜닝 전 추론 결과 ===')

for _, row in test.head(3).iterrows():
    prompt = create_prompt(row['input'], few_shot=True)
    output = pipe(
        prompt,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    print(f'입력: {row["input"]}')
    print(f'출력: {output[0]["generated_text"][len(prompt):].strip()}')
    print()

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'eos_token_id', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


=== 파인튜닝 전 추론 결과 ===


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem

입력: 녀뮨넒뭅 만죡숭러윤 효템뤼에오. 푸싸눼 옰면 콕 츄쩐학꼬 싶은 콧쉰웨오. 췌꾜윕뉘댜! ㅎㅎ 당음웨 또 옭 컷 갗았요.
출력: 녀석이 엊그저께 엊그저께 엊그저께 엊그저께 엊그저께 엊그저께 엊그



[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


입력: 풀룐투갸 엎코, 좀식또 업읍머, 윌뱐 잎츔민든릿 샤있샤윔엡 위썬 호뗄첨렴 관뤽갉 찰 앉 뙨는 누뀜뮈넬오. 까썽뷔갚 떨여쳐옵.
출력: 풀룐투갸 엎코, 좀식또 업읍머, 윌뱐 잎츔민든릿 샤있샤윔엡 위썬 호뗄첨렴 관뤽갉

입력: 쥔차 붉찐졀행욘. 삶먼섶 멂묽럿턴 혹텔 중웨 쬐약위였습뉜따. 칙어뉜쥐 샤쨩윈쥐 쩨끄윈할 땝붇텄 찐쩔함 1됴 없섣규욤. 3인 옌악학꼬 츄갚 쿰맥카찔 껄졔헷눈테 슉켠 츄까료 오청두링닒 쿤쩡썅 쑤컨 4께팎웨 첵콩인 않 된댜녜욥. 2쉬갼 졍돈 웨쭐핥꼬 닸싯 틀역칼 태 몄 효쉰냐, 2뿐 얘약햐쥐 않앝냥 묽엽봇썼어오(췌큼인핥 떼와 갇툰 뷴). 츄같큼 컬줴했교 깟트 껼젠 네억 뮨쟝됴 뽀없둘렸는뎁또 깟둡변홅 몃 뻔윈진 걺젤한 쉭갼, 푼, 촘꺄쥣 뮬엽봇쒸떪라규오. 걸쳬한 엉쑤중 칙쩝 짱즛쉰 닻음, 홋숫 찰못 져겼탸 한맏띠 핥셨엷오. 싻괏 한 먀딕 엾읊셧규오. 엎윅까 엾꼬 샹닿힐 풀쾨햇쑵뉘닯. 구릭코 냉쟝교엣써 섕선 퓔린냇 낢오. 묽엣써툐 핑륀맡 낮옆욤…
출력: 맛있고 양 많아요!!

Input: 얆얆 얆얆 얆얆 얆얆 얆얆 얆



In [ ]:
# 데이터셋에 프롬프트 적용 후 HuggingFace Dataset 형식으로 변환

def format_chat_template(row):
    prompt = create_prompt(row['input'], row['output'])
    row['text'] = prompt  # ← text 컬럼 추가
    return row

# train_data, val_data를 Dataset으로 변환
train_dataset = Dataset.from_pandas(train_data[['input', 'output']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_data[['input', 'output']].reset_index(drop=True))

train_dataset = train_dataset.map(format_chat_template, batched=False)
val_dataset = val_dataset.map(format_chat_template, batched=False)

Map:   0%|          | 0/10136 [00:00<?, ? examples/s]

Map:   0%|          | 0/1127 [00:00<?, ? examples/s]

In [ ]:
# 프롬프트 변환 결과 및 데이터셋 구조 확인

print(train_dataset[0])
print(train_dataset.column_names)

{'input': '넒무 좋았서용. 뿌됴, 싣섧동~~!! 톡 깥콥팔올ㅎㅎ', 'output': '너무 좋았어요. 뷰도, 시설도~~!! 또 가고파요ㅎㅎ', 'text': '당신은 난독화된 한국어 리뷰를 자연스럽고 정확한 문장으로 복원하는 전문가이다.\n반드시 복원된 문장만 출력하라.\n\nInput: 넒무 좋았서용. 뿌됴, 싣섧동~~!! 톡 깥콥팔올ㅎㅎ\nOutput: 너무 좋았어요. 뷰도, 시설도~~!! 또 가고파요ㅎㅎ<eos>'}
['input', 'output', 'text']


In [ ]:
print(tokenizer(train_dataset[0]['text'], return_tensors='pt')['input_ids'].shape)

torch.Size([1, 110])


In [ ]:
# LoRA 설정 후 SFT 학습 실행 및 어댑터 저장

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.train()

training_args = SFTConfig(
    output_dir='./results',
    num_train_epochs=1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,

    fp16=False,
    bf16=False,

    optim='paged_adamw_8bit',

    logging_steps=20,
    eval_strategy='steps',
    eval_steps=150,
    save_strategy='epoch',

    report_to='none',
    dataset_text_field='text',
    max_length=128,
)

print("fp16:", training_args.fp16)
print("bf16:", training_args.bf16)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset.shuffle(seed=42),
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()

ADAPTER_MODEL = 'lora_adapter_2b'
trainer.model.save_pretrained(ADAPTER_MODEL)
tokenizer.save_pretrained(ADAPTER_MODEL)

/tmp/ipykernel_1158/2405429284.py:21: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


fp16: False
bf16: False


Adding EOS to train dataset:   0%|          | 0/10136 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10136 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1127 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1127 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
150,2.516843,2.536278,2.482269,0.587278,143667.000000


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
150,2.516843,2.536278,2.482269,0.587278,143667.000000
300,2.448977,2.444804,2.496471,0.595755,287762.000000
450,2.373557,2.398481,2.446063,0.600523,431684.000000
600,2.373702,2.360853,2.441004,0.603029,575555.000000
750,2.349085,2.329256,2.396923,0.605553,719360.000000
900,2.243067,2.309962,2.360792,0.607133,863547.000000
1050,2.340856,2.297831,2.369177,0.607520,1007696.000000
1200,2.196658,2.291519,2.351796,0.608898,1152010.000000
1267,2.322260,2.291250,2.348322,0.609016,1216318.000000


('lora_adapter_2b/tokenizer_config.json', 'lora_adapter_2b/tokenizer.json')

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, 'lora_adapter_2b')

In [ ]:
# 저장된 LoRA 어댑터 파일 목록 확인

import os
os.listdir('lora_adapter_2b')

['tokenizer_config.json',
 'README.md',
 'tokenizer.json',
 'adapter_config.json',
 'adapter_model.safetensors']

In [ ]:
# 파인튜닝된 모델로 추론 파이프라인 생성

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer
)

In [ ]:
# 문자 단위 F1 스코어 계산 함수 정의

from collections import Counter

def char_f1(pred, answer):
    pred_chars = Counter(pred)
    ans_chars = Counter(answer)

    common = sum((pred_chars & ans_chars).values())

    if common == 0:
        return 0.0

    precision = common / sum(pred_chars.values())
    recall = common / sum(ans_chars.values())

    return 2 * (precision * recall) / (precision + recall)

In [ ]:
# 검증 데이터 30개로 파인튜닝 후 모델 성능 평가

val_preds = []
val_sample = val_data.head(30)

for _, row in val_sample.iterrows():
    prompt = create_prompt(row['input'])
    output = pipe(
        prompt,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    val_preds.append(
        output[0]['generated_text'][len(prompt):].strip()
    )

scores = [char_f1(pred, ans) for pred, ans in zip(val_preds, val_sample['output'])]
print(f'Validation Char F1: {sum(scores)/len(scores):.4f}')

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transforme

Validation Char F1: 0.4155


In [ ]:
# GPU 메모리 캐시 정리

import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
restored_reviews = []

test_prompts = [create_prompt(text) for text in test['input'].tolist()]

for start in range(0, len(test_prompts), 4):
    batch_prompts = test_prompts[start:start+4]

    outputs = pipe(
        batch_prompts,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        batch_size=4
    )

    for prompt, output in zip(batch_prompts, outputs):
        generated_text = output[0]['generated_text'] if isinstance(output, list) else output['generated_text']
        result = generated_text[len(prompt):].strip()
        restored_reviews.append(result)

    if len(restored_reviews) % 100 == 0:
        print(f"{len(restored_reviews)}개 완료")

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

100개 완료


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

200개 완료


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

...

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

1500개 완료


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

1600개 완료


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

In [ ]:
# 추론 결과를 제출 파일로 저장

submission = pd.read_csv("/content/drive/MyDrive/dataset/sample_submission.csv", encoding="utf-8-sig")

submission["output"] = restored_reviews

submission.to_csv("submission_llm.csv", index=False, encoding="utf-8-sig")

print("submission_llm.csv 저장 완료!")

submission_llm.csv 저장 완료!


In [ ]:
# 성능 개선 실험 - 디코딩 전략 비교 실험

strategies = [
    {"label": "Greedy", "do_sample": False},
    {"label": "Beam Search (4)", "do_sample": False, "num_beams": 4},
    {"label": "Top-p Sampling", "do_sample": True, "top_p": 0.9, "temperature": 0.7},
]

for strategy in strategies:
    val_preds = []
    val_sample = val_data.head(30)

    for _, row in val_sample.iterrows():
        prompt = create_prompt(row['input'])
        output = pipe(
            prompt,
            max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            **{k: v for k, v in strategy.items() if k != "label"}
        )
        val_preds.append(output[0]['generated_text'][len(prompt):].strip())

    scores = [char_f1(pred, ans) for pred, ans in zip(val_preds, val_sample['output'])]
    print(f"{strategy['label']}: {sum(scores)/len(scores):.4f}")

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Greedy: 0.4155


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Beam Search (4): 0.4506


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

Top-p Sampling: 0.4280


In [ ]:
output = pipe(
    prompt,
    max_new_tokens=50,
    do_sample=False,
    num_beams=4,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
import shutil
shutil.make_archive('lora_adapter_2b', 'zip', 'lora_adapter_2b')
shutil.make_archive('results', 'zip', 'results')

'/content/results.zip'